# 06 — Expected Length-of-Stay Model Development and Validation

## Purpose

This notebook develops and validates retrospective case-mix-adjusted length-of-stay models using the governed outputs from Notebooks 01–05.

It implements:

- Explicit leakage-control and feature contracts
- Hospital-held-out cross-validation for the current one-year source
- Statewide, APR-DRG, and APR-DRG × severity mean and median baselines
- Ridge, Poisson, Tweedie, Random Forest, and XGBoost candidates
- Identical validation folds across candidate models
- MAE, median absolute error, RMSE, aggregate actual-to-expected calibration, and runtime comparison
- Subgroup performance diagnostics
- `120 +` LOS sensitivity diagnostics
- Hospital-identity and discharge-disposition sensitivity analysis
- Provisional model shortlisting
- A versioned prediction-output contract for Notebook 07
- Machine-readable modeling outputs and business-readable documentation

## Modeling Objective

The target is `los_days_lower_bound`. The task is retrospective expected-LOS estimation, not admission-time clinical prediction.

The primary model excludes LOS-derived fields, charges, estimated costs, descriptive peer expectations, hospital identity, patient disposition, and procedure code. Hospital identity and disposition are evaluated only as sensitivity variables because they can absorb operational differences the project is intended to measure.

`120 +` LOS values remain observable lower bounds of 120 days and are retained in the primary development population.

## Primary Validation Strategy

Because the current implemented source contains one year, the primary validation design holds out entire hospitals. Every candidate uses the same hospital-held-out folds.

Future-year stability is not demonstrated here.

## Model Selection Boundary

Notebook 06 produces a provisional shortlist. Notebook 07 will decide whether the incremental ML value justifies added complexity, freeze the final model version, refit it, score the source snapshot, and create `model_predicted_los_days`.

`peer_expected_los_days` and `model_predicted_los_days` remain separate analytical concepts.

## Modeling Dependencies

Recommended environment additions:

```bash
pip install numpy scikit-learn xgboost
```

The notebook does not install packages at runtime.

## 1. Imports

In [1]:
from pathlib import Path
import json
import platform
import time

import duckdb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import PoissonRegressor, Ridge, TweedieRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    import xgboost
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    xgboost = None
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)


## 2. Project Paths and Configuration

In [2]:
def find_project_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "docs" / "project_charter.md").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
AUDIT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
CATALOG_DIR = PROJECT_ROOT / "outputs" / "metric_catalog"
SCHEMA_DIR = PROJECT_ROOT / "outputs" / "star_schema"
PHYSICAL_DIR = PROJECT_ROOT / "outputs" / "physical_model"
PHYSICAL_TABLE_DIR = PHYSICAL_DIR / "tables"
BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "peer_benchmarks"
BENCHMARK_TABLE_DIR = BENCHMARK_DIR / "tables"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "expected_los_model"
WORK_DIR = OUTPUT_DIR / "_work"
DOCS_DIR = PROJECT_ROOT / "docs"
WORK_DB_PATH = WORK_DIR / "expected_los_modeling.duckdb"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
DEVELOPMENT_MAX_ROWS = 250_000
N_GROUP_FOLDS = 3
SENSITIVITY_MAX_ROWS = 100_000
N_SENSITIVITY_FOLDS = 2
ONE_HOT_MIN_FREQUENCY = 25
PREDICTION_FLOOR_DAYS = 0.01
RANDOM_FOREST_TREES = 80
XGBOOST_TREES = 350
REPORTING_MIN_N = 11

assert XGBOOST_AVAILABLE, (
    "XGBoost is required for the governed gradient-boosting candidate. "
    "Install it with: pip install xgboost"
)

environment_versions = pd.DataFrame([
    ["python", platform.python_version()],
    ["pandas", pd.__version__],
    ["numpy", np.__version__],
    ["duckdb", duckdb.__version__],
    ["scikit-learn", sklearn.__version__],
    ["xgboost", xgboost.__version__],
], columns=["component", "version"])

print("Project root:", PROJECT_ROOT.name)
print("Development max rows:", f"{DEVELOPMENT_MAX_ROWS:,}")
print("Hospital-held-out folds:", N_GROUP_FOLDS)
print("Minimum exportable subgroup size:", REPORTING_MIN_N)
display(environment_versions)


Project root: 04_hospital_operations_powerbi
Development max rows: 250,000
Hospital-held-out folds: 3
Minimum exportable subgroup size: 11


,component,version
0,python,3.14.3
1,pandas,3.0.5
2,numpy,2.5.1
3,duckdb,1.5.5
4,scikit-learn,1.9.0
5,xgboost,3.4.1


## 3. Load Upstream Validation Outputs

In [3]:
input_files = {
    "file_metadata": AUDIT_DIR / "file_metadata.csv",
    "catalog_validation": CATALOG_DIR / "validation_results.csv",
    "schema_validation": SCHEMA_DIR / "schema_validation_results.csv",
    "physical_validation": PHYSICAL_DIR / "physical_validation_results.csv",
    "physical_parquet_validation": PHYSICAL_DIR / "parquet_validation.csv",
    "benchmark_validation": BENCHMARK_DIR / "benchmark_validation_results.csv",
    "benchmark_parquet_validation": BENCHMARK_DIR / "parquet_validation.csv",
}

table_paths = {
    "FactDischarge": BENCHMARK_TABLE_DIR / "FactDischarge.parquet",
    "DimHospital": PHYSICAL_TABLE_DIR / "DimHospital.parquet",
    "DimService": PHYSICAL_TABLE_DIR / "DimService.parquet",
    "DimCaseMix": PHYSICAL_TABLE_DIR / "DimCaseMix.parquet",
    "DimDiagnosis": PHYSICAL_TABLE_DIR / "DimDiagnosis.parquet",
    "DimProcedure": PHYSICAL_TABLE_DIR / "DimProcedure.parquet",
    "DimPatientSegment": PHYSICAL_TABLE_DIR / "DimPatientSegment.parquet",
    "DimPayer": PHYSICAL_TABLE_DIR / "DimPayer.parquet",
    "DimAdmissionContext": PHYSICAL_TABLE_DIR / "DimAdmissionContext.parquet",
}

missing_files = [
    str(path)
    for path in [*input_files.values(), *table_paths.values()]
    if not path.exists()
]

assert not missing_files, (
    "Required upstream outputs are missing:\n" + "\n".join(missing_files)
)

inputs = {name: pd.read_csv(path) for name, path in input_files.items()}
file_metadata = inputs["file_metadata"]
catalog_validation = inputs["catalog_validation"]
schema_validation = inputs["schema_validation"]
physical_validation = inputs["physical_validation"]
physical_parquet_validation = inputs["physical_parquet_validation"]
benchmark_validation = inputs["benchmark_validation"]
benchmark_parquet_validation = inputs["benchmark_parquet_validation"]

display(benchmark_validation)

,validation_test,passed,details
0,Notebook 02 validation passed,True,NaN
1,Notebook 03 validation passed,True,NaN
2,Notebook 04 physical validation passed,True,NaN
3,Input benchmark columns were empty,True,NaN
4,Benchmark context preserves fact row count,True,2125754 vs 2125754
5,Benchmarked fact preserves fact row count,True,2125754 vs 2125754
6,Non-benchmark fact columns are unchanged,True,0 input-only; 0 output-only
7,Benchmark selection logic has no violations,True,0
8,Direct mathematical reconciliation passes,True,12 sample comparisons
9,Fact schema remains unchanged,True,NaN


## 4. Upstream Commit Gates

In [4]:
def normalize_boolean(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

for label, frame in {
    "Notebook 02": catalog_validation,
    "Notebook 03": schema_validation,
    "Notebook 04 physical model": physical_validation,
    "Notebook 04 Parquet": physical_parquet_validation,
    "Notebook 05 benchmark": benchmark_validation,
    "Notebook 05 Parquet": benchmark_parquet_validation,
}.items():
    assert normalize_boolean(frame["passed"]).fillna(False).all(), (
        f"{label} validation results contain failures."
    )

assert len(file_metadata) == 1
SOURCE_SHA256 = str(file_metadata.loc[0, "sha256"]).strip()
SOURCE_SNAPSHOT_ID = SOURCE_SHA256[:16]
assert SOURCE_SHA256

print("Notebook 02–05 validation and Parquet commit gates passed.")
print("Source snapshot ID:", SOURCE_SNAPSHOT_ID)

Notebook 02–05 validation and Parquet commit gates passed.
Source snapshot ID: d69e4b9e47fd2992


## 5. Primary Feature and Leakage Contract

In [5]:
PRIMARY_FEATURES = [
    "apr_drg_code",
    "apr_mdc_code",
    "medical_surgical_classification",
    "apr_severity_code",
    "apr_mortality_risk",
    "ccsr_diagnosis_code",
    "age_group",
    "gender",
    "payer_group",
    "admission_type_group",
    "ed_indicator_group",
]

FORBIDDEN_PRIMARY_FEATURES = {
    "source_record_key",
    "hospital_key",
    "permanent_facility_id",
    "facility_name",
    "hospital_service_area",
    "hospital_county",
    "disposition_group",
    "ccsr_procedure_code",
    "los_days_lower_bound",
    "is_top_coded_los",
    "is_valid_los",
    "total_charges",
    "total_costs",
    "peer_expected_los_days",
    "los_peer_comparison_n",
    "los_peer_benchmark_level",
    "peer_expected_estimated_cost",
    "cost_peer_comparison_n",
    "cost_peer_benchmark_level",
}

SENSITIVITY_FEATURES = {
    "Primary": PRIMARY_FEATURES,
    "Primary + Hospital": PRIMARY_FEATURES + ["hospital_identity_sensitivity"],
    "Primary + Disposition": PRIMARY_FEATURES + ["disposition_group"],
    "Primary + Hospital + Disposition": PRIMARY_FEATURES + [
        "hospital_identity_sensitivity", "disposition_group"
    ],
}

assert not (set(PRIMARY_FEATURES) & FORBIDDEN_PRIMARY_FEATURES)

primary_feature_rows = [
    ["apr_drg_code", "DimService", "Primary feature", True, "Core discharge clinical/service classification."],
    ["apr_mdc_code", "DimService", "Primary feature", True, "Broad diagnostic-category context."],
    ["medical_surgical_classification", "DimService", "Primary feature", True, "Medical-versus-surgical context."],
    ["apr_severity_code", "DimCaseMix", "Primary feature", True, "Governed severity-of-illness case-mix measure."],
    ["apr_mortality_risk", "DimCaseMix", "Primary feature", True, "Released mortality-risk case-complexity context."],
    ["ccsr_diagnosis_code", "DimDiagnosis", "Primary feature", True, "Diagnosis-category case-mix context."],
    ["age_group", "DimPatientSegment", "Primary feature", True, "Approved age case-mix context."],
    ["gender", "DimPatientSegment", "Primary feature", True, "Released demographic context."],
    ["payer_group", "DimPayer", "Primary feature", True, "Approved payer-mix context."],
    ["admission_type_group", "DimAdmissionContext", "Primary feature", True, "Admission context."],
    ["ed_indicator_group", "DimAdmissionContext", "Primary feature", True, "Emergency-department admission context."],
]

excluded_feature_details = {
    "source_record_key": ("FactDischarge", "Technical key only; never a predictive feature."),
    "hospital_key": ("FactDischarge", "Used only to define hospital-held-out folds and sensitivity identity."),
    "permanent_facility_id": ("DimHospital", "Hospital identity can absorb facility performance differences."),
    "facility_name": ("DimHospital", "Hospital identity can absorb facility performance differences."),
    "hospital_service_area": ("DimHospital", "Hospital geography is reserved for diagnostics, not the primary model."),
    "hospital_county": ("DimHospital", "Hospital geography is reserved for diagnostics, not the primary model."),
    "disposition_group": ("DimAdmissionContext", "Downstream of the inpatient stay; sensitivity only."),
    "ccsr_procedure_code": ("DimProcedure", "May reflect care delivered during the stay."),
    "los_days_lower_bound": ("FactDischarge", "Model target; prohibited as an input feature."),
    "is_top_coded_los": ("FactDischarge", "Derived from the target; diagnostics only."),
    "is_valid_los": ("FactDischarge", "Target-availability flag used only for population eligibility."),
    "total_charges": ("FactDischarge", "Downstream financial/resource-use field."),
    "total_costs": ("FactDischarge", "Downstream financial/resource-use field."),
    "peer_expected_los_days": ("FactDischarge", "Descriptive peer expectation must remain analytically independent."),
    "los_peer_comparison_n": ("FactDischarge", "Peer-benchmark diagnostic, not a predictive feature."),
    "los_peer_benchmark_level": ("FactDischarge", "Peer-benchmark diagnostic, not a predictive feature."),
    "peer_expected_estimated_cost": ("FactDischarge", "Descriptive financial benchmark, not a predictive feature."),
    "cost_peer_comparison_n": ("FactDischarge", "Peer-benchmark diagnostic, not a predictive feature."),
    "cost_peer_benchmark_level": ("FactDischarge", "Peer-benchmark diagnostic, not a predictive feature."),
}

feature_specification = pd.DataFrame(
    primary_feature_rows
    + [[name, *excluded_feature_details[name][:1], "Forbidden primary", False, excluded_feature_details[name][1]]
       for name in sorted(FORBIDDEN_PRIMARY_FEATURES)]
    + [["hospital_identity_sensitivity", "Derived from hospital_key", "Sensitivity only", False,
        "Evaluates hospital-identity dependence outside primary validation."]],
    columns=["feature_name", "source_table", "role", "included_primary", "rationale"],
)

assert set(PRIMARY_FEATURES).issubset(set(feature_specification["feature_name"]))
assert FORBIDDEN_PRIMARY_FEATURES.issubset(set(feature_specification["feature_name"]))
display(feature_specification)


,feature_name,source_table,role,included_primary,rationale
0,apr_drg_code,DimService,Primary feature,True,Core discharge clinical/service classification.
1,apr_mdc_code,DimService,Primary feature,True,Broad diagnostic-category context.
2,medical_surgical_classification,DimService,Primary feature,True,Medical-versus-surgical context.
3,apr_severity_code,DimCaseMix,Primary feature,True,Governed severity-of-illness case-mix measure.
4,apr_mortality_risk,DimCaseMix,Primary feature,True,Released mortality-risk case-complexity context.
5,ccsr_diagnosis_code,DimDiagnosis,Primary feature,True,Diagnosis-category case-mix context.
6,age_group,DimPatientSegment,Primary feature,True,Approved age case-mix context.
7,gender,DimPatientSegment,Primary feature,True,Released demographic context.
8,payer_group,DimPayer,Primary feature,True,Approved payer-mix context.
9,admission_type_group,DimAdmissionContext,Primary feature,True,Admission context.


## 6. Candidate Model Specification

In [6]:
model_candidate_specification = pd.DataFrame([
    ["STATEWIDE_MEDIAN", "Transparent baseline", "Conditional median baseline", True],
    ["APR_DRG_MEDIAN", "Transparent baseline", "Conditional median baseline", True],
    ["APR_DRG_SEVERITY_MEDIAN", "Transparent median baseline", "Conditional median baseline", True],
    ["STATEWIDE_MEAN", "Transparent mean baseline","Conditional mean baseline",True],
    ["APR_DRG_MEAN", "Transparent mean baseline", "Conditional mean baseline", True],
    ["APR_DRG_SEVERITY_MEAN", "Strong transparent mean baseline","Conditional mean baseline",True],
    ["RIDGE_REGRESSION", "Statistical / ML", "Raw-scale conditional mean approximation", True],
    ["POISSON_REGRESSION", "Statistical / ML", "Conditional mean", True],
    ["TWEEDIE_REGRESSION", "Statistical / ML", "Conditional mean", True],
    ["RANDOM_FOREST", "Statistical / ML", "Conditional mean approximation", True],
    ["XGBOOST", "Statistical / ML", "Conditional mean approximation", True],
    ["QUANTILE_P50_P90", "Optional deferred", "Conditional quantiles", False],
], columns=["model_name", "candidate_type", "target_estimand", "included"])

display(model_candidate_specification)

,model_name,candidate_type,target_estimand,included
0,STATEWIDE_MEDIAN,Transparent baseline,Conditional median baseline,True
1,APR_DRG_MEDIAN,Transparent baseline,Conditional median baseline,True
2,APR_DRG_SEVERITY_MEDIAN,Transparent median baseline,Conditional median baseline,True
3,STATEWIDE_MEAN,Transparent mean baseline,Conditional mean baseline,True
4,APR_DRG_MEAN,Transparent mean baseline,Conditional mean baseline,True
5,APR_DRG_SEVERITY_MEAN,Strong transparent mean baseline,Conditional mean baseline,True
6,RIDGE_REGRESSION,Statistical / ML,Raw-scale conditional mean approximation,True
7,POISSON_REGRESSION,Statistical / ML,Conditional mean,True
8,TWEEDIE_REGRESSION,Statistical / ML,Conditional mean,True
9,RANDOM_FOREST,Statistical / ML,Conditional mean approximation,True


## 7. Load Analytical Tables and Build Modeling Source

In [7]:
def sql_path_literal(path):
    return "'" + Path(path).resolve().as_posix().replace("'", "''") + "'"

def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'

if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()

con = duckdb.connect(database=str(WORK_DB_PATH))

for table_name, table_path in table_paths.items():
    con.execute(
        f'''CREATE OR REPLACE VIEW {quote_identifier(table_name + "Input")} AS
            SELECT * FROM read_parquet({sql_path_literal(table_path)})'''
    )

fact_row_count = int(con.sql(
    'SELECT COUNT(*) AS row_count FROM "FactDischargeInput"'
).df().loc[0, "row_count"])

con.execute('''
    CREATE OR REPLACE TABLE modeling_source AS
    SELECT
        f.source_record_key,
        f.hospital_key,
        h.permanent_facility_id,
        h.facility_name,
        h.hospital_service_area,
        h.hospital_county,
        svc.apr_drg_code,
        svc.apr_mdc_code,
        svc.medical_surgical_classification,
        cm.apr_severity_code,
        cm.apr_mortality_risk,
        dx.ccsr_diagnosis_code,
        px.ccsr_procedure_code,
        ps.age_group,
        ps.gender,
        ps.race,
        ps.ethnicity,
        pay.payer_group,
        ac.admission_type_group,
        ac.disposition_group,
        ac.ed_indicator_group,
        f.los_days_lower_bound,
        f.is_top_coded_los,
        f.is_valid_los,
        f.peer_expected_los_days,
        f.los_peer_comparison_n,
        f.los_peer_benchmark_level
    FROM "FactDischargeInput" AS f
    LEFT JOIN "DimHospitalInput" AS h ON f.hospital_key = h.hospital_key
    LEFT JOIN "DimServiceInput" AS svc ON f.service_key = svc.service_key
    LEFT JOIN "DimCaseMixInput" AS cm ON f.case_mix_key = cm.case_mix_key
    LEFT JOIN "DimDiagnosisInput" AS dx ON f.diagnosis_key = dx.diagnosis_key
    LEFT JOIN "DimProcedureInput" AS px ON f.procedure_key = px.procedure_key
    LEFT JOIN "DimPatientSegmentInput" AS ps ON f.patient_segment_key = ps.patient_segment_key
    LEFT JOIN "DimPayerInput" AS pay ON f.payer_key = pay.payer_key
    LEFT JOIN "DimAdmissionContextInput" AS ac ON f.admission_context_key = ac.admission_context_key
''')

modeling_source_row_count = int(con.sql(
    'SELECT COUNT(*) AS row_count FROM modeling_source'
).df().loc[0, "row_count"])

source_record_key_validation = con.sql('''
    SELECT
        COUNT(*) AS row_count,
        COUNT(source_record_key) AS nonnull_key_n,
        COUNT(DISTINCT source_record_key) AS distinct_key_n
    FROM modeling_source
''').df()
source_key_row = source_record_key_validation.iloc[0]
source_record_key_valid = (
    int(source_key_row["row_count"])
    == int(source_key_row["nonnull_key_n"])
    == int(source_key_row["distinct_key_n"])
)

assert modeling_source_row_count == fact_row_count
assert source_record_key_valid
print("Benchmarked fact rows:", f"{fact_row_count:,}")

Benchmarked fact rows: 2,125,754


## 8. Modeling Population and Deterministic Development Sample

In [8]:
modeling_population_summary = con.sql('''
    SELECT
        COUNT(*) AS total_fact_rows,
        SUM(CASE WHEN is_valid_los = 1 AND los_days_lower_bound IS NOT NULL AND hospital_key <> 0 THEN 1 ELSE 0 END) AS eligible_modeling_rows,
        COUNT(DISTINCT CASE WHEN is_valid_los = 1 AND los_days_lower_bound IS NOT NULL AND hospital_key <> 0 THEN hospital_key END) AS eligible_hospital_n,
        SUM(CASE WHEN is_valid_los = 1 AND los_days_lower_bound IS NOT NULL AND hospital_key <> 0 AND is_top_coded_los = 1 THEN 1 ELSE 0 END) AS eligible_top_coded_n,
        AVG(CASE WHEN is_valid_los = 1 AND los_days_lower_bound IS NOT NULL AND hospital_key <> 0 THEN CAST(los_days_lower_bound AS DOUBLE) END) AS eligible_mean_los,
        MEDIAN(CASE WHEN is_valid_los = 1 AND los_days_lower_bound IS NOT NULL AND hospital_key <> 0 THEN CAST(los_days_lower_bound AS DOUBLE) END) AS eligible_median_los
    FROM modeling_source
''').df()

modeling_population_summary.insert(0, "source_snapshot_id", SOURCE_SNAPSHOT_ID)
eligible_modeling_n = int(modeling_population_summary.loc[0, "eligible_modeling_rows"])
eligible_hospital_n = int(modeling_population_summary.loc[0, "eligible_hospital_n"])
assert eligible_modeling_n > 0
assert eligible_hospital_n >= N_GROUP_FOLDS

development_n = min(DEVELOPMENT_MAX_ROWS, eligible_modeling_n)
development_df = con.sql(f'''
    SELECT *
    FROM modeling_source
    WHERE is_valid_los = 1
      AND los_days_lower_bound IS NOT NULL
      AND hospital_key <> 0
    ORDER BY md5(CAST(source_record_key AS VARCHAR) || '-' || CAST({RANDOM_SEED} AS VARCHAR))
    LIMIT {development_n}
''').df()

assert len(development_df) == development_n
development_df["target_los"] = development_df["los_days_lower_bound"].astype(float)
development_df["hospital_identity_sensitivity"] = development_df["hospital_key"].astype(str)

categorical_needed = sorted(set(
    PRIMARY_FEATURES + [
        "disposition_group", "hospital_service_area", "hospital_county",
        "facility_name", "ccsr_procedure_code"
    ]
))
for column in categorical_needed:
    development_df[column] = (
        development_df[column]
        .astype("string")
        .fillna("Unknown / Not Available")
        .astype(str)
    )

development_df["apr_severity_code"] = (
    development_df["apr_severity_code"]
    .astype("string")
    .fillna("Unknown / Not Available")
    .astype(str)
)

display(modeling_population_summary)
print("Development rows:", f"{len(development_df):,}")
print("Development hospitals:", development_df["hospital_key"].nunique())

,source_snapshot_id,total_fact_rows,eligible_modeling_rows,eligible_hospital_n,eligible_top_coded_n,eligible_mean_los,eligible_median_los
0,d69e4b9e47fd2992,2125754,2120421.0,207,2272.0,5.785011,3.0


Development rows: 250,000
Development hospitals: 207


## 9. Development Validation

In [9]:
development_validation = pd.DataFrame([
    ["Development sample is nonempty", len(development_df) > 0, str(len(development_df))],
    ["Development target is strictly positive", (development_df["target_los"] > 0).all(), f"min={development_df['target_los'].min()}"],
    ["Development hospital keys are resolved", (development_df["hospital_key"] != 0).all(), ""],
    ["Primary features contain no forbidden fields", not bool(set(PRIMARY_FEATURES) & FORBIDDEN_PRIMARY_FEATURES), ""],
    ["Primary feature columns exist", set(PRIMARY_FEATURES).issubset(development_df.columns), ""],
], columns=["validation_test", "passed", "details"])

display(development_validation)
assert development_validation["passed"].all()

,validation_test,passed,details
0,Development sample is nonempty,True,250000
1,Development target is strictly positive,True,min=1.0
2,Development hospital keys are resolved,True,
3,Primary features contain no forbidden fields,True,
4,Primary feature columns exist,True,


## 10. Modeling Utilities

In [10]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=ONE_HOT_MIN_FREQUENCY,
            sparse_output=True,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=ONE_HOT_MIN_FREQUENCY,
            sparse=True,
            dtype=np.float32,
        )


def make_model_pipeline(model_name, feature_columns):
    preprocessor = ColumnTransformer(
        [("categorical", make_one_hot_encoder(), feature_columns)],
        remainder="drop",
        sparse_threshold=1.0,
    )

    if model_name == "RIDGE_REGRESSION":
        estimator = Ridge(alpha=10.0, solver="lsqr")
    elif model_name == "POISSON_REGRESSION":
        estimator = PoissonRegressor(alpha=1.0, max_iter=300, tol=1e-6)
    elif model_name == "TWEEDIE_REGRESSION":
        estimator = TweedieRegressor(power=1.5, alpha=1.0, link="log", max_iter=300, tol=1e-6)
    elif model_name == "RANDOM_FOREST":
        estimator = RandomForestRegressor(
            n_estimators=RANDOM_FOREST_TREES,
            max_depth=18,
            min_samples_leaf=10,
            max_features="sqrt",
            max_samples=0.75,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    elif model_name == "XGBOOST":
        estimator = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=XGBOOST_TREES,
            learning_rate=0.05,
            max_depth=8,
            min_child_weight=10,
            subsample=0.80,
            colsample_bytree=0.80,
            reg_lambda=1.0,
            tree_method="hist",
            eval_metric="mae",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return Pipeline([("preprocessor", preprocessor), ("model", estimator)])


def clip_predictions(predictions):
    return np.maximum(np.asarray(predictions, dtype=float), PREDICTION_FLOOR_DAYS)


def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = clip_predictions(y_pred)
    actual_sum = float(np.sum(y_true))
    predicted_sum = float(np.sum(y_pred))
    ae = actual_sum / predicted_sum if predicted_sum > 0 else np.nan
    return {
        "n": len(y_true),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "median_absolute_error": float(median_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "actual_mean": float(np.mean(y_true)),
        "predicted_mean": float(np.mean(y_pred)),
        "actual_sum": actual_sum,
        "predicted_sum": predicted_sum,
        "actual_to_expected_ratio": float(ae),
        "calibration_error_abs": float(abs(ae - 1.0)),
    }


def baseline_predictions(train_df, validation_df):
    statewide = float(train_df["target_los"].median())
    statewide_pred = np.full(len(validation_df), statewide, dtype=float)

    drg_lookup = train_df.groupby("apr_drg_code", dropna=False)["target_los"].median()
    drg_pred = validation_df["apr_drg_code"].map(drg_lookup).fillna(statewide).to_numpy(float)

    drg_sev_lookup = train_df.groupby(
        ["apr_drg_code", "apr_severity_code"], dropna=False
    )["target_los"].median()
    keys = pd.MultiIndex.from_frame(validation_df[["apr_drg_code", "apr_severity_code"]])
    drg_sev_pred = drg_sev_lookup.reindex(keys).to_numpy(float)
    drg_sev_pred = np.where(np.isnan(drg_sev_pred), drg_pred, drg_sev_pred)

    statewide_mean = float(train_df["target_los"].mean())
    statewide_mean_pred = np.full(
        len(validation_df),
        statewide_mean,
        dtype=float
    )

    drg_mean_lookup = (
        train_df
        .groupby("apr_drg_code", dropna=False)["target_los"]
        .mean()
    )

    drg_mean_pred = (
        validation_df["apr_drg_code"]
        .map(drg_mean_lookup)
        .fillna(statewide_mean)
        .to_numpy(float)
    )

    drg_sev_mean_lookup = (
        train_df.groupby(
            ["apr_drg_code", "apr_severity_code"],
            dropna=False,
        )["target_los"]
        .mean()
    )

    keys = pd.MultiIndex.from_frame(
        validation_df[
            ["apr_drg_code", "apr_severity_code"]
        ]
    )

    drg_sev_mean_pred = (
        drg_sev_mean_lookup
        .reindex(keys)
        .to_numpy(float)
    )

    drg_sev_mean_pred = np.where(
        np.isnan(drg_sev_mean_pred),
        drg_mean_pred,
        drg_sev_mean_pred
    )

    return {
        "STATEWIDE_MEDIAN": statewide_pred,
        "APR_DRG_MEDIAN": drg_pred,
        "APR_DRG_SEVERITY_MEDIAN": drg_sev_pred,
        "STATEWIDE_MEAN": statewide_mean_pred,
        "APR_DRG_MEAN": drg_mean_pred,
        "APR_DRG_SEVERITY_MEAN": drg_sev_mean_pred
    }

## 11. Hospital-Held-Out Fold Assignment

In [11]:
group_cv = GroupKFold(n_splits=N_GROUP_FOLDS)
fold_assignment = np.full(len(development_df), -1, dtype=int)
fold_indices = []

for fold_number, (train_index, validation_index) in enumerate(
    group_cv.split(development_df, groups=development_df["hospital_key"]), start=1
):
    fold_assignment[validation_index] = fold_number
    fold_indices.append((fold_number, train_index, validation_index))

assert (fold_assignment > 0).all()
development_df["validation_fold"] = fold_assignment

fold_hospital_summary = (
    development_df.groupby("validation_fold")
    .agg(
        validation_rows=("source_record_key", "size"),
        validation_hospitals=("hospital_key", "nunique"),
        validation_mean_los=("target_los", "mean"),
        validation_median_los=("target_los", "median"),
    )
    .reset_index()
)

display(fold_hospital_summary)

,validation_fold,validation_rows,validation_hospitals,validation_mean_los,validation_median_los
0,1,83335,69,5.692074,3.0
1,2,83331,69,5.988540,3.0
2,3,83334,69,5.665395,3.0


## 12. Identical-Fold Candidate Evaluation

In [12]:
baseline_model_names = [
    "STATEWIDE_MEDIAN",
    "APR_DRG_MEDIAN",
    "APR_DRG_SEVERITY_MEDIAN",
    "STATEWIDE_MEAN",
    "APR_DRG_MEAN",
    "APR_DRG_SEVERITY_MEAN",
]
statistical_model_names = [
    "RIDGE_REGRESSION",
    "POISSON_REGRESSION",
    "TWEEDIE_REGRESSION",
    "RANDOM_FOREST",
    "XGBOOST",
]
all_model_names = baseline_model_names + statistical_model_names

oof_predictions = {
    name: np.full(len(development_df), np.nan, dtype=float)
    for name in all_model_names
}
fold_metrics_rows = []

for fold_number, train_index, validation_index in fold_indices:
    train_df = development_df.iloc[train_index].copy()
    validation_df = development_df.iloc[validation_index].copy()
    y_train = train_df["target_los"].to_numpy(float)
    y_validation = validation_df["target_los"].to_numpy(float)

    print(f"Fold {fold_number}: train={len(train_df):,}, validation={len(validation_df):,}")

    baseline_start = time.perf_counter()
    fold_baselines = baseline_predictions(train_df, validation_df)
    baseline_elapsed = time.perf_counter() - baseline_start

    for model_name, predictions in fold_baselines.items():
        predictions = clip_predictions(predictions)
        oof_predictions[model_name][validation_index] = predictions
        metrics = evaluate_predictions(y_validation, predictions)
        metrics.update({
            "fold": fold_number,
            "model_name": model_name,
            "candidate_type": "Transparent baseline",
            "fit_predict_seconds": baseline_elapsed / len(fold_baselines),
        })
        fold_metrics_rows.append(metrics)

    for model_name in statistical_model_names:
        start = time.perf_counter()
        pipeline = make_model_pipeline(model_name, PRIMARY_FEATURES)
        pipeline.fit(train_df[PRIMARY_FEATURES], y_train)
        predictions = clip_predictions(pipeline.predict(validation_df[PRIMARY_FEATURES]))
        elapsed = time.perf_counter() - start
        oof_predictions[model_name][validation_index] = predictions
        metrics = evaluate_predictions(y_validation, predictions)
        metrics.update({
            "fold": fold_number,
            "model_name": model_name,
            "candidate_type": "Statistical / ML",
            "fit_predict_seconds": elapsed,
        })
        fold_metrics_rows.append(metrics)
        print(
            f"  {model_name}: MAE={metrics['mae']:.4f}, "
            f"A/E={metrics['actual_to_expected_ratio']:.4f}, seconds={elapsed:.1f}"
        )

cross_validation_fold_metrics = pd.DataFrame(fold_metrics_rows)
display(cross_validation_fold_metrics)

Fold 1: train=166,665, validation=83,335


  RIDGE_REGRESSION: MAE=3.2858, A/E=1.0223, seconds=2.0


  POISSON_REGRESSION: MAE=3.7887, A/E=0.9851, seconds=3.9


  TWEEDIE_REGRESSION: MAE=3.9069, A/E=1.0189, seconds=2.0


  RANDOM_FOREST: MAE=3.3980, A/E=1.0043, seconds=3.5


...

  POISSON_REGRESSION: MAE=3.8550, A/E=0.9784, seconds=2.1


  TWEEDIE_REGRESSION: MAE=3.9642, A/E=1.0143, seconds=2.1


  RANDOM_FOREST: MAE=3.4717, A/E=0.9888, seconds=3.6


  XGBOOST: MAE=3.2337, A/E=0.9965, seconds=5.0


,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs,fold,model_name,candidate_type,fit_predict_seconds
0,83335,3.834235,2.000000,8.895601,5.692074,3.000000,474349.0,250005.000000,1.897358,0.897358,1,STATEWIDE_MEDIAN,Transparent baseline,0.016084
1,83335,3.258475,1.000000,7.645744,5.692074,3.887754,474349.0,323986.000000,1.464103,0.464103,1,APR_DRG_MEDIAN,Transparent baseline,0.016084
2,83335,3.005298,1.000000,7.161479,5.692074,4.258901,474349.0,354915.500000,1.336512,0.336512,1,APR_DRG_SEVERITY_MEDIAN,Transparent baseline,0.016084
3,83335,4.550757,3.826964,8.479543,5.692074,5.826964,474349.0,485590.067411,0.976851,0.023149,1,STATEWIDE_MEAN,Transparent baseline,0.016084
4,83335,3.555564,1.996540,7.302816,5.692074,5.581091,474349.0,465100.183397,1.019886,0.019886,1,APR_DRG_MEAN,Transparent baseline,0.016084
5,83335,3.195606,1.553824,6.883041,5.692074,5.536398,474349.0,461375.723462,1.028119,0.028119,1,APR_DRG_SEVERITY_MEAN,Transparent baseline,0.016084
6,83335,3.285814,1.682809,6.929809,5.692074,5.567643,474349.0,463979.545329,1.022349,0.022349,1,RIDGE_REGRESSION,Statistical / ML,1.985859
7,83335,3.788716,2.448328,7.742961,5.692074,5.778351,474349.0,481538.918065,0.985069,0.014931,1,POISSON_REGRESSION,Statistical / ML,3.882453
8,83335,3.906940,2.585934,8.003929,5.692074,5.586586,474349.0,465558.182863,1.018882,0.018882,1,TWEEDIE_REGRESSION,Statistical / ML,1.999752
9,83335,3.398015,2.052795,7.143247,5.692074,5.667677,474349.0,472315.823351,1.004305,0.004305,1,RANDOM_FOREST,Statistical / ML,3.480703


## 13. OOF Prediction Validation and Overall Model Comparison

In [13]:
oof_prediction_validation = pd.DataFrame([
    {
        "model_name": name,
        "missing_prediction_n": int(np.isnan(oof_predictions[name]).sum()),
        "nonpositive_prediction_n": int((oof_predictions[name] <= 0).sum()),
    }
    for name in all_model_names
])
oof_prediction_validation["passed"] = (
    oof_prediction_validation["missing_prediction_n"].eq(0)
    & oof_prediction_validation["nonpositive_prediction_n"].eq(0)
)
assert oof_prediction_validation["passed"].all()

y_all = development_df["target_los"].to_numpy(float)
runtime_summary = cross_validation_fold_metrics.groupby("model_name")["fit_predict_seconds"].sum().to_dict()
candidate_type_lookup = model_candidate_specification.set_index("model_name")["candidate_type"].to_dict()

rows = []
for name in all_model_names:
    metrics = evaluate_predictions(y_all, oof_predictions[name])
    metrics.update({
        "model_name": name,
        "candidate_type": candidate_type_lookup[name],
        "total_cv_seconds": float(runtime_summary.get(name, np.nan)),
    })
    rows.append(metrics)

model_comparison = pd.DataFrame(rows)
for metric in ["mae", "median_absolute_error", "rmse", "calibration_error_abs"]:
    model_comparison[f"{metric}_rank"] = model_comparison[metric].rank(method="min")

rank_cols = ["mae_rank", "median_absolute_error_rank", "rmse_rank", "calibration_error_abs_rank"]
model_comparison["multi_metric_rank_score"] = model_comparison[rank_cols].mean(axis=1)

strong_baseline_mae = float(model_comparison.loc[
    model_comparison["model_name"].eq("APR_DRG_SEVERITY_MEAN"), "mae"
].iloc[0])

model_comparison["mae_improvement_vs_strong_baseline_pct"] = (
    (strong_baseline_mae - model_comparison["mae"]) / strong_baseline_mae * 100.0
)
model_comparison = model_comparison.sort_values(
    ["multi_metric_rank_score", "mae"]
).reset_index(drop=True)

display(oof_prediction_validation)
display(model_comparison)

,model_name,missing_prediction_n,nonpositive_prediction_n,passed
0,STATEWIDE_MEDIAN,0,0,True
1,APR_DRG_MEDIAN,0,0,True
2,APR_DRG_SEVERITY_MEDIAN,0,0,True
3,STATEWIDE_MEAN,0,0,True
4,APR_DRG_MEAN,0,0,True
5,APR_DRG_SEVERITY_MEAN,0,0,True
6,RIDGE_REGRESSION,0,0,True
7,POISSON_REGRESSION,0,0,True
8,TWEEDIE_REGRESSION,0,0,True
9,RANDOM_FOREST,0,0,True


,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs,model_name,candidate_type,total_cv_seconds,mae_rank,median_absolute_error_rank,rmse_rank,calibration_error_abs_rank,multi_metric_rank_score,mae_improvement_vs_strong_baseline_pct
0,250000,3.336727,1.634036,7.061390,5.782,5.783082,1445500.0,1.445771e+06,0.999813,1.871257e-04,APR_DRG_SEVERITY_MEAN,Strong transparent mean baseline,0.049535,4.0,3.0,2.0,4.0,3.25,0.000000
1,250000,3.284980,1.658697,6.959015,5.782,5.776028,1445500.0,1.444007e+06,1.001034,1.033852e-03,XGBOOST,Statistical / ML,14.506381,2.0,4.0,1.0,7.0,3.50,1.550821
2,250000,3.087932,1.000000,7.296578,5.782,4.472464,1445500.0,1.118116e+06,1.292800,2.927997e-01,APR_DRG_SEVERITY_MEDIAN,Transparent median baseline,0.049535,1.0,1.0,4.0,9.0,3.75,7.456247
3,250000,3.424587,1.734176,7.128632,5.782,5.783009,1445500.0,1.445752e+06,0.999825,1.745372e-04,RIDGE_REGRESSION,Statistical / ML,5.939483,5.0,5.0,3.0,3.0,4.00,-2.633142
4,250000,3.512404,2.082932,7.361639,5.782,5.782320,1445500.0,1.445580e+06,0.999945,5.540278e-05,RANDOM_FOREST,Statistical / ML,10.151408,6.0,7.0,5.0,2.0,5.00,-5.264961
5,250000,3.331900,1.000000,7.792970,5.782,4.077964,1445500.0,1.019491e+06,1.417864,4.178644e-01,APR_DRG_MEDIAN,Transparent baseline,0.049535,3.0,1.0,7.0,10.0,5.25,0.144650
6,250000,3.682133,2.096085,7.488386,5.782,5.785758,1445500.0,1.446439e+06,0.999351,6.494561e-04,APR_DRG_MEAN,Transparent mean baseline,0.049535,7.0,8.0,6.0,5.0,6.50,-10.351657
7,250000,3.886158,2.483806,8.026967,5.782,5.777531,1445500.0,1.444383e+06,1.000774,7.735059e-04,POISSON_REGRESSION,Statistical / ML,7.762485,8.0,9.0,8.0,6.0,7.75,-16.466190
8,250000,4.618605,3.678734,8.752644,5.782,5.782002,1445500.0,1.445501e+06,1.000000,3.681377e-07,STATEWIDE_MEAN,Transparent mean baseline,0.049535,11.0,11.0,10.0,1.0,8.25,-38.417257
9,250000,3.923728,2.000000,9.182673,5.782,3.000000,1445500.0,7.500000e+05,1.927333,9.273333e-01,STATEWIDE_MEDIAN,Transparent baseline,0.049535,9.0,6.0,11.0,11.0,9.25,-17.592135


## 14. Provisional Statistical / ML Shortlist

In [14]:
statistical_comparison = (
    model_comparison.loc[model_comparison["candidate_type"].eq("Statistical / ML")]
    .sort_values(["multi_metric_rank_score", "mae"])
    .reset_index(drop=True)
)
assert not statistical_comparison.empty

model_shortlist = statistical_comparison.head(3).copy()
model_shortlist.insert(0, "shortlist_rank", np.arange(1, len(model_shortlist) + 1))
PROVISIONAL_MODEL_NAME = str(model_shortlist.loc[0, "model_name"])
PROVISIONAL_BEATS_STRONG_BASELINE = float(model_shortlist.loc[0, "mae"]) < strong_baseline_mae

display(model_shortlist)
print("Provisional statistical/ML candidate:", PROVISIONAL_MODEL_NAME)
print("Lower MAE than APR-DRG × severity mean baseline:", PROVISIONAL_BEATS_STRONG_BASELINE)

,shortlist_rank,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs,model_name,candidate_type,total_cv_seconds,mae_rank,median_absolute_error_rank,rmse_rank,calibration_error_abs_rank,multi_metric_rank_score,mae_improvement_vs_strong_baseline_pct
0,1,250000,3.284980,1.658697,6.959015,5.782,5.776028,1445500.0,1.444007e+06,1.001034,0.001034,XGBOOST,Statistical / ML,14.506381,2.0,4.0,1.0,7.0,3.5,1.550821
1,2,250000,3.424587,1.734176,7.128632,5.782,5.783009,1445500.0,1.445752e+06,0.999825,0.000175,RIDGE_REGRESSION,Statistical / ML,5.939483,5.0,5.0,3.0,3.0,4.0,-2.633142
2,3,250000,3.512404,2.082932,7.361639,5.782,5.782320,1445500.0,1.445580e+06,0.999945,0.000055,RANDOM_FOREST,Statistical / ML,10.151408,6.0,7.0,5.0,2.0,5.0,-5.264961


Provisional statistical/ML candidate: XGBOOST
Lower MAE than APR-DRG × severity mean baseline: True


### Interpretation

The shortlist is provisional. A lower validation error alone does not make a model production-ready. Notebook 07 must consider calibration, subgroup behavior, sensitivity, runtime, maintainability, and whether the incremental value over the transparent baseline justifies the complexity.

## 15. Subgroup Performance Diagnostics

In [15]:
evaluation_df = development_df[[
    "source_record_key", "hospital_key", "hospital_service_area",
    "apr_drg_code", "apr_severity_code", "age_group", "payer_group",
    "admission_type_group", "is_top_coded_los", "validation_fold", "target_los"
]].copy()
evaluation_df["strong_baseline_prediction"] = oof_predictions["APR_DRG_SEVERITY_MEAN"]
evaluation_df["provisional_model_prediction"] = oof_predictions[PROVISIONAL_MODEL_NAME]

SUBGROUP_COLUMNS = [
    "hospital_key", "hospital_service_area", "apr_drg_code", "apr_severity_code",
    "age_group", "payer_group", "admission_type_group"
]

def subgroup_rows(dataframe, group_column, prediction_column, model_name):
    rows = []
    for group_value, group_df in dataframe.groupby(group_column, dropna=False):
        metrics = evaluate_predictions(group_df["target_los"], group_df[prediction_column])
        rows.append({
            "model_name": model_name,
            "subgroup_dimension": group_column,
            "subgroup_value": str(group_value),
            **metrics,
        })
    return rows

rows = []
for column in SUBGROUP_COLUMNS:
    rows.extend(subgroup_rows(
        evaluation_df, column, "strong_baseline_prediction", "APR_DRG_SEVERITY_MEAN"
    ))
    rows.extend(subgroup_rows(
        evaluation_df, column, "provisional_model_prediction", PROVISIONAL_MODEL_NAME
    ))

subgroup_performance_internal = pd.DataFrame(rows)
suppressed_subgroup_mask = subgroup_performance_internal["n"].lt(REPORTING_MIN_N)
subgroup_performance = (
    subgroup_performance_internal.loc[~suppressed_subgroup_mask]
    .reset_index(drop=True)
)
subgroup_suppression_summary = (
    subgroup_performance_internal.loc[suppressed_subgroup_mask]
    .groupby(["model_name", "subgroup_dimension"], as_index=False)
    .size()
    .rename(columns={"size": "suppressed_group_n"})
)

display(subgroup_performance.head(30))
display(subgroup_suppression_summary)
print("Exportable subgroup diagnostic rows:", len(subgroup_performance))
print("Suppressed low-volume subgroup rows:", int(suppressed_subgroup_mask.sum()))


,model_name,subgroup_dimension,subgroup_value,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs
0,APR_DRG_SEVERITY_MEAN,hospital_key,1,4333,3.746374,1.997275,6.853330,6.406647,6.682389,27760.0,28954.789501,0.958736,0.041264
1,APR_DRG_SEVERITY_MEAN,hospital_key,2,2628,2.871550,1.807398,4.394921,4.545662,6.109336,11946.0,16055.334788,0.744052,0.255948
2,APR_DRG_SEVERITY_MEAN,hospital_key,3,133,7.419086,6.116803,10.872480,7.939850,11.241269,1056.0,1495.088797,0.706313,0.293687
3,APR_DRG_SEVERITY_MEAN,hospital_key,5,223,2.449393,1.461268,3.598251,2.717489,4.796084,606.0,1069.526835,0.566606,0.433394
4,APR_DRG_SEVERITY_MEAN,hospital_key,6,371,6.381739,3.736916,10.156525,9.495957,10.413178,3523.0,3863.289169,0.911917,0.088083
5,APR_DRG_SEVERITY_MEAN,hospital_key,7,990,2.659238,1.625770,4.475005,4.725253,5.111166,4678.0,5060.053856,0.924496,0.075504
6,APR_DRG_SEVERITY_MEAN,hospital_key,8,1546,3.091108,1.614671,5.460681,5.546572,5.683279,8575.0,8786.349075,0.975946,0.024054
7,APR_DRG_SEVERITY_MEAN,hospital_key,9,739,3.038127,1.889546,4.821478,5.454668,5.705490,4031.0,4216.356888,0.956039,0.043961
8,APR_DRG_SEVERITY_MEAN,hospital_key,10,553,3.124244,1.714612,7.082263,5.092224,5.766847,2816.0,3189.066299,0.883017,0.116983
9,APR_DRG_SEVERITY_MEAN,hospital_key,11,117,2.584088,2.263084,3.474788,4.102564,5.062159,480.0,592.272547,0.810438,0.189562


,model_name,subgroup_dimension,suppressed_group_n
0,APR_DRG_SEVERITY_MEAN,apr_drg_code,11
1,APR_DRG_SEVERITY_MEAN,hospital_key,3
2,XGBOOST,apr_drg_code,11
3,XGBOOST,hospital_key,3


Exportable subgroup diagnostic rows: 1118
Suppressed low-volume subgroup rows: 28


## 16. LOS Top-Code Sensitivity

In [16]:
top_code_sensitivity_rows = []
populations = {
    "All eligible development rows": evaluation_df,
    "Exclude 120+ top-coded LOS": evaluation_df.loc[evaluation_df["is_top_coded_los"].eq(0)],
    "120+ top-coded LOS only": evaluation_df.loc[evaluation_df["is_top_coded_los"].eq(1)],
}
prediction_sets = {
    "APR_DRG_SEVERITY_MEAN": "strong_baseline_prediction",
    PROVISIONAL_MODEL_NAME: "provisional_model_prediction",
}

for population_name, population_df in populations.items():
    if population_df.empty:
        continue
    for model_name, prediction_column in prediction_sets.items():
        metrics = evaluate_predictions(population_df["target_los"], population_df[prediction_column])
        top_code_sensitivity_rows.append({
            "population": population_name,
            "model_name": model_name,
            **metrics,
        })

top_code_sensitivity = pd.DataFrame(top_code_sensitivity_rows)
display(top_code_sensitivity)

,population,model_name,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs
0,All eligible development rows,APR_DRG_SEVERITY_MEAN,250000,3.336727,1.634036,7.061390,5.782000,5.783082,1445500.0,1.445771e+06,0.999813,0.000187
1,All eligible development rows,XGBOOST,250000,3.284980,1.658697,6.959015,5.782000,5.776028,1445500.0,1.444007e+06,1.001034,0.001034
2,Exclude 120+ top-coded LOS,APR_DRG_SEVERITY_MEAN,249727,3.245305,1.632454,6.401455,5.657138,5.753290,1412740.0,1.436752e+06,0.983287,0.016713
3,Exclude 120+ top-coded LOS,XGBOOST,249727,3.194396,1.655905,6.303537,5.657138,5.745335,1412740.0,1.434765e+06,0.984649,0.015351
4,120+ top-coded LOS only,APR_DRG_SEVERITY_MEAN,273,86.964855,97.109375,90.427088,120.000000,33.035145,32760.0,9.018595e+03,3.632495,2.632495
5,120+ top-coded LOS only,XGBOOST,273,86.146694,96.236368,89.446328,120.000000,33.853306,32760.0,9.241953e+03,3.544705,2.544705


## 17. Hospital Identity and Disposition Sensitivity

The primary validation excludes hospital identity and disposition. This secondary analysis uses random row-level folds so hospital identity can recur across training and validation. It is not evidence of unseen-hospital generalization. Its purpose is to quantify how much apparent predictive performance changes when the model can absorb hospital identity and/or downstream disposition.

In [17]:
sensitivity_n = min(SENSITIVITY_MAX_ROWS, len(development_df))
sensitivity_df = development_df.sample(
    n=sensitivity_n, random_state=RANDOM_SEED
).reset_index(drop=True)

row_cv = KFold(
    n_splits=N_SENSITIVITY_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

sensitivity_rows = []
for feature_set_name, feature_columns in SENSITIVITY_FEATURES.items():
    oof = np.full(len(sensitivity_df), np.nan, dtype=float)
    total_seconds = 0.0

    for train_index, validation_index in row_cv.split(sensitivity_df):
        train_df = sensitivity_df.iloc[train_index]
        validation_df = sensitivity_df.iloc[validation_index]
        model = make_model_pipeline(PROVISIONAL_MODEL_NAME, feature_columns)
        start = time.perf_counter()
        model.fit(train_df[feature_columns], train_df["target_los"].to_numpy(float))
        oof[validation_index] = clip_predictions(model.predict(validation_df[feature_columns]))
        total_seconds += time.perf_counter() - start

    assert not np.isnan(oof).any()
    metrics = evaluate_predictions(sensitivity_df["target_los"], oof)
    sensitivity_rows.append({
        "model_name": PROVISIONAL_MODEL_NAME,
        "validation_design": "Random row-level KFold sensitivity",
        "feature_set": feature_set_name,
        "feature_count": len(feature_columns),
        "total_cv_seconds": total_seconds,
        **metrics,
    })

hospital_disposition_sensitivity = pd.DataFrame(sensitivity_rows)
primary_sensitivity_mae = float(hospital_disposition_sensitivity.loc[
    hospital_disposition_sensitivity["feature_set"].eq("Primary"), "mae"
].iloc[0])
hospital_disposition_sensitivity["mae_change_vs_primary_pct"] = (
    (hospital_disposition_sensitivity["mae"] - primary_sensitivity_mae)
    / primary_sensitivity_mae * 100.0
)

display(hospital_disposition_sensitivity)

,model_name,validation_design,feature_set,feature_count,total_cv_seconds,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs,mae_change_vs_primary_pct
0,XGBOOST,Random row-level KFold sensitivity,Primary,11,4.492684,100000,3.299692,1.633836,7.011062,5.76654,5.764329,576654.0,576432.934278,1.000384,0.000384,0.000000
1,XGBOOST,Random row-level KFold sensitivity,Primary + Hospital,12,5.093098,100000,3.257630,1.630573,6.906167,5.76654,5.757397,576654.0,575739.739074,1.001588,0.001588,-1.274735
2,XGBOOST,Random row-level KFold sensitivity,Primary + Disposition,12,5.056363,100000,3.167058,1.518140,6.849733,5.76654,5.764051,576654.0,576405.078693,1.000432,0.000432,-4.019579
3,XGBOOST,Random row-level KFold sensitivity,Primary + Hospital + Disposition,13,5.394326,100000,3.121325,1.505972,6.740386,5.76654,5.756144,576654.0,575614.380006,1.001806,0.001806,-5.405567


## 18. Model Development Decision Summary

In [18]:
provisional_row = model_comparison.loc[
    model_comparison["model_name"].eq(PROVISIONAL_MODEL_NAME)
].iloc[0]
strong_baseline_row = model_comparison.loc[
    model_comparison["model_name"].eq("APR_DRG_SEVERITY_MEAN")
].iloc[0]

model_development_decision = pd.DataFrame([
    ["Provisional statistical/ML candidate", PROVISIONAL_MODEL_NAME, "Best statistical/ML candidate under the multi-metric hospital-held-out ranking."],
    ["Provisional candidate MAE", f"{float(provisional_row['mae']):.6f}", "Hospital-held-out OOF MAE."],
    ["Strong baseline MAE", f"{float(strong_baseline_row['mae']):.6f}", "APR-DRG × severity mean OOF MAE."],
    ["MAE improvement vs strong baseline", f"{float(provisional_row['mae_improvement_vs_strong_baseline_pct']):.4f}%", "Positive means lower MAE than the strong baseline."],
    ["Provisional aggregate A/E", f"{float(provisional_row['actual_to_expected_ratio']):.6f}", "Aggregate actual LOS divided by aggregate OOF predicted LOS."],
    ["Ready for Notebook 07 review", "Yes", "Notebook 07 decides whether incremental value justifies complexity."],
], columns=["decision_item", "value", "interpretation"])

display(model_development_decision)

,decision_item,value,interpretation
0,Provisional statistical/ML candidate,XGBOOST,Best statistical/ML candidate under the multi-metric hospital-held-out ranking.
1,Provisional candidate MAE,3.284980,Hospital-held-out OOF MAE.
2,Strong baseline MAE,3.336727,APR-DRG × severity mean OOF MAE.
3,MAE improvement vs strong baseline,1.5508%,Positive means lower MAE than the strong baseline.
4,Provisional aggregate A/E,1.001034,Aggregate actual LOS divided by aggregate OOF predicted LOS.
5,Ready for Notebook 07 review,Yes,Notebook 07 decides whether incremental value justifies complexity.


## 19. Prediction Output Contract for Notebook 07

In [19]:
prediction_output_specification = pd.DataFrame([
    ["source_record_key", "Whole number", True, "Technical row key scoped only to the source snapshot; not a patient or durable longitudinal discharge identifier."],
    ["source_snapshot_id", "Text", True, "Identifier derived from the audited source snapshot."],
    ["model_version", "Text", True, "Version identifier for the finalized expected-LOS model."],
    ["scoring_timestamp", "Timestamp", True, "Timestamp at which the source snapshot was scored."],
    ["model_predicted_los_days", "Decimal number", True, "Raw-day model prediction from the finalized conditional-mean expected-LOS model."],
    ["prediction_status", "Text", True, "Coverage/scoring status for the source row."],
    ["missing_prediction_reason", "Text", True, "Reason prediction is unavailable when the row is not scored."],
    ["model_predicted_p50_los", "Decimal number", False, "Optional validated conditional-median LOS estimate."],
    ["model_predicted_p90_los", "Decimal number", False, "Optional validated conditional-P90 LOS estimate."],
], columns=["column_name", "data_type", "required", "description"])

display(prediction_output_specification)

,column_name,data_type,required,description
0,source_record_key,Whole number,True,Technical row key scoped only to the source snapshot; not a patient or durable longitudinal discharge identifier.
1,source_snapshot_id,Text,True,Identifier derived from the audited source snapshot.
2,model_version,Text,True,Version identifier for the finalized expected-LOS model.
3,scoring_timestamp,Timestamp,True,Timestamp at which the source snapshot was scored.
4,model_predicted_los_days,Decimal number,True,Raw-day model prediction from the finalized conditional-mean expected-LOS model.
5,prediction_status,Text,True,Coverage/scoring status for the source row.
6,missing_prediction_reason,Text,True,Reason prediction is unavailable when the row is not scored.
7,model_predicted_p50_los,Decimal number,False,Optional validated conditional-median LOS estimate.
8,model_predicted_p90_los,Decimal number,False,Optional validated conditional-P90 LOS estimate.


## 20. Modeling Validation Gates

In [20]:
modeling_validation_results = pd.DataFrame([
    ["Notebook 02 validation passed", normalize_boolean(catalog_validation["passed"]).fillna(False).all(), ""],
    ["Notebook 03 validation passed", normalize_boolean(schema_validation["passed"]).fillna(False).all(), ""],
    ["Notebook 04 physical validation passed", normalize_boolean(physical_validation["passed"]).fillna(False).all(), ""],
    ["Notebook 04 Parquet validation passed", normalize_boolean(physical_parquet_validation["passed"]).fillna(False).all(), ""],
    ["Notebook 05 benchmark validation passed", normalize_boolean(benchmark_validation["passed"]).fillna(False).all(), ""],
    ["Notebook 05 Parquet validation passed", normalize_boolean(benchmark_parquet_validation["passed"]).fillna(False).all(), ""],
    ["Modeling source preserves fact row count", modeling_source_row_count == fact_row_count, f"{modeling_source_row_count} vs {fact_row_count}"],
    ["Source record key is complete and unique", source_record_key_valid, str(source_record_key_validation.to_dict("records")[0])],
    ["Feature specification covers all primary and forbidden fields", (
        set(PRIMARY_FEATURES).issubset(set(feature_specification["feature_name"]))
        and FORBIDDEN_PRIMARY_FEATURES.issubset(set(feature_specification["feature_name"]))
    ), str(len(feature_specification))],
    ["Development validation passed", development_validation["passed"].all(), ""],
    ["Hospital-held-out folds cover every development row", (development_df["validation_fold"] > 0).all(), f"{N_GROUP_FOLDS} folds"],
    ["All candidate OOF predictions are complete and positive", oof_prediction_validation["passed"].all(), ""],
    ["All governed candidate families were evaluated", set(all_model_names).issubset(set(model_comparison["model_name"])), "|".join(all_model_names)],
    ["Provisional shortlist exists", not model_shortlist.empty, "|".join(model_shortlist["model_name"].astype(str))],
    ["Exported subgroup diagnostics meet minimum reporting size", (
        not subgroup_performance.empty and subgroup_performance["n"].ge(REPORTING_MIN_N).all()
    ), f"minimum n={REPORTING_MIN_N}; rows={len(subgroup_performance)}"],
    ["Low-volume subgroup diagnostics are summarized without group values", (
        int(subgroup_suppression_summary["suppressed_group_n"].sum())
        == int(suppressed_subgroup_mask.sum())
    ), str(int(suppressed_subgroup_mask.sum()))],
    ["Top-code sensitivity produced", not top_code_sensitivity.empty, str(len(top_code_sensitivity))],
    ["Hospital/disposition sensitivity produced", not hospital_disposition_sensitivity.empty, str(len(hospital_disposition_sensitivity))],
    ["Prediction contract separates modeled from peer expected LOS", (
        "model_predicted_los_days" in set(prediction_output_specification["column_name"])
        and "peer_expected_los_days" not in set(prediction_output_specification["column_name"])
    ), ""],
], columns=["validation_test", "passed", "details"])

display(modeling_validation_results)
assert modeling_validation_results["passed"].all(), (
    "Notebook 06 failed one or more modeling validation gates."
)
print("All Notebook 06 modeling validation gates passed.")


,validation_test,passed,details
0,Notebook 02 validation passed,True,
1,Notebook 03 validation passed,True,
2,Notebook 04 physical validation passed,True,
3,Notebook 04 Parquet validation passed,True,
4,Notebook 05 benchmark validation passed,True,
5,Notebook 05 Parquet validation passed,True,
6,Modeling source preserves fact row count,True,2125754 vs 2125754
7,Source record key is complete and unique,True,"{'row_count': 2125754, 'nonnull_key_n': 2125754, 'distinct_key_n': 2125754}"
8,Feature specification covers all primary and forbidden fields,True,31
9,Development validation passed,True,


All Notebook 06 modeling validation gates passed.


## 21. Export Machine-Readable Modeling Artifacts

In [21]:
export_frames = {
    "environment_versions.csv": environment_versions,
    "source_record_key_validation.csv": source_record_key_validation,
    "modeling_population_summary.csv": modeling_population_summary,
    "feature_specification.csv": feature_specification,
    "model_candidate_specification.csv": model_candidate_specification,
    "fold_hospital_summary.csv": fold_hospital_summary,
    "cross_validation_fold_metrics.csv": cross_validation_fold_metrics,
    "oof_prediction_validation.csv": oof_prediction_validation,
    "model_comparison.csv": model_comparison,
    "model_shortlist.csv": model_shortlist,
    "subgroup_performance.csv": subgroup_performance,
    "subgroup_suppression_summary.csv": subgroup_suppression_summary,
    "top_code_sensitivity.csv": top_code_sensitivity,
    "hospital_disposition_sensitivity.csv": hospital_disposition_sensitivity,
    "model_development_decision.csv": model_development_decision,
    "prediction_output_specification.csv": prediction_output_specification,
    "modeling_validation_results.csv": modeling_validation_results,
}

export_manifest_rows = []
for file_name, dataframe in export_frames.items():
    output_path = OUTPUT_DIR / file_name
    dataframe.to_csv(output_path, index=False)
    assert output_path.exists() and output_path.stat().st_size > 0
    export_manifest_rows.append({
        "file_name": file_name,
        "artifact_type": "Model development output",
        "row_count": len(dataframe),
        "file_size_mb": round(output_path.stat().st_size / (1024 ** 2), 4),
        "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
    })

export_manifest = pd.DataFrame(export_manifest_rows)
display(export_manifest)


,file_name,artifact_type,row_count,file_size_mb,output_path
0,environment_versions.csv,Model development output,6,0.0001,outputs/expected_los_model/environment_versions.csv
1,source_record_key_validation.csv,Model development output,1,0.0001,outputs/expected_los_model/source_record_key_validation.csv
2,modeling_population_summary.csv,Model development output,1,0.0002,outputs/expected_los_model/modeling_population_summary.csv
3,feature_specification.csv,Model development output,31,0.0032,outputs/expected_los_model/feature_specification.csv
4,model_candidate_specification.csv,Model development output,12,0.0009,outputs/expected_los_model/model_candidate_specification.csv
5,fold_hospital_summary.csv,Model development output,3,0.0002,outputs/expected_los_model/fold_hospital_summary.csv
6,cross_validation_fold_metrics.csv,Model development output,33,0.0069,outputs/expected_los_model/cross_validation_fold_metrics.csv
7,oof_prediction_validation.csv,Model development output,11,0.0003,outputs/expected_los_model/oof_prediction_validation.csv
8,model_comparison.csv,Model development output,11,0.0028,outputs/expected_los_model/model_comparison.csv
9,model_shortlist.csv,Model development output,3,0.0010,outputs/expected_los_model/model_shortlist.csv


### Portfolio Export Boundary

Notebook 06 exports aggregate model-development diagnostics rather than row-level out-of-fold prediction records. Notebook 07 may create controlled versioned scoring artifacts for downstream analytical use.

## 22. Generate `docs/expected_los_modeling.md`

In [22]:
def dataframe_to_markdown(dataframe, columns):
    selected = dataframe.loc[:, columns].fillna("").astype(str)
    def escape_value(value):
        return value.replace("|", r"\|").replace("\n", " ")
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = [
        "| " + " | ".join(escape_value(value) for value in row) + " |"
        for row in selected.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])

comparison_markdown = dataframe_to_markdown(model_comparison, [
    "model_name", "candidate_type", "mae", "median_absolute_error", "rmse",
    "actual_to_expected_ratio", "calibration_error_abs", "total_cv_seconds",
    "mae_improvement_vs_strong_baseline_pct", "multi_metric_rank_score"
])
shortlist_markdown = dataframe_to_markdown(model_shortlist, [
    "shortlist_rank", "model_name", "mae", "median_absolute_error", "rmse",
    "actual_to_expected_ratio", "mae_improvement_vs_strong_baseline_pct"
])
sensitivity_markdown = dataframe_to_markdown(hospital_disposition_sensitivity, [
    "feature_set", "n", "mae", "median_absolute_error", "rmse",
    "actual_to_expected_ratio", "mae_change_vs_primary_pct"
])
environment_markdown = dataframe_to_markdown(environment_versions, ["component", "version"])
suppression_markdown = dataframe_to_markdown(subgroup_suppression_summary, [
    "model_name", "subgroup_dimension", "suppressed_group_n"
])
validation_markdown = dataframe_to_markdown(modeling_validation_results, [
    "validation_test", "passed", "details"
])

expected_los_document = "\n".join([
    "# Expected LOS Model Development — Hospital Operations & Cost Efficiency",
    "",
    "## Purpose",
    "",
    "This document summarizes retrospective case-mix-adjusted LOS model development and hospital-held-out validation.",
    "",
    "## Source Snapshot",
    "",
    f"- Source snapshot ID: `{SOURCE_SNAPSHOT_ID}`",
    f"- Eligible modeling rows: {eligible_modeling_n:,}",
    f"- Development rows: {len(development_df):,}",
    f"- Development hospitals: {development_df['hospital_key'].nunique():,}",
    "- Predictions can be joined by the complete, unique `source_record_key` scoped to this snapshot.",
    "",
    "## Runtime Versions",
    "",
    environment_markdown,
    "",
    "## Primary Validation Design",
    "",
    f"The notebook uses {N_GROUP_FOLDS}-fold hospital-held-out cross-validation.",
    "",
    "Hospital identity is used only to define held-out groups and is not a primary predictive feature.",
    "",
    "## Primary Features",
    "",
    *[f"- `{feature}`" for feature in PRIMARY_FEATURES],
    "",
    "## Candidate Model Comparison",
    "",
    comparison_markdown,
    "",
    "## Provisional Statistical / ML Shortlist",
    "",
    shortlist_markdown,
    "",
    "## Provisional Candidate",
    "",
    f"`{PROVISIONAL_MODEL_NAME}`",
    "",
    "This is a development-stage shortlist result, not a final production-model version.",
    "",
    "## Hospital / Disposition Sensitivity",
    "",
    sensitivity_markdown,
    "",
    "This sensitivity uses random row-level folds and must not be interpreted as unseen-hospital generalization.",
    "",
    "## Privacy-Safe Subgroup Reporting",
    "",
    f"Subgroup results with fewer than {REPORTING_MIN_N} discharges are omitted from the detailed export. Only aggregate suppressed-group counts are retained below.",
    "",
    suppression_markdown,
    "",
    "## LOS Top-Coding",
    "",
    "The primary development population retains `120 +` LOS records as observable 120-day lower bounds. Separate diagnostics quantify their influence.",
    "",
    "## Validation Results",
    "",
    validation_markdown,
    "",
    "## Limitations",
    "",
    "- The current source contains one implemented year, so future-year stability is not demonstrated.",
    "- The model is retrospective rather than an admission-time clinical prediction model.",
    "- Unmeasured clinical and structural differences may remain.",
    "- Model predictions support operational screening and do not establish causality or preventability.",
    "- Descriptive peer expectations and model predictions remain separate analytical concepts.",
    "",
    "## Next Step",
    "",
    "Notebook 07 will make the final model-version decision, refit the approved approach, create versioned scoring outputs, and prepare `model_predicted_los_days` for Power BI.",
    "",
])

EXPECTED_LOS_DOCUMENT_PATH = DOCS_DIR / "expected_los_modeling.md"
EXPECTED_LOS_DOCUMENT_PATH.write_text(expected_los_document, encoding="utf-8")
assert EXPECTED_LOS_DOCUMENT_PATH.exists() and EXPECTED_LOS_DOCUMENT_PATH.stat().st_size > 0
print("Expected LOS documentation created:", EXPECTED_LOS_DOCUMENT_PATH.relative_to(PROJECT_ROOT))


Expected LOS documentation created: docs\expected_los_modeling.md


## 23. Finalize Export Manifest

In [23]:
document_manifest_row = pd.DataFrame([{
    "file_name": EXPECTED_LOS_DOCUMENT_PATH.name,
    "artifact_type": "Documentation",
    "row_count": pd.NA,
    "file_size_mb": round(EXPECTED_LOS_DOCUMENT_PATH.stat().st_size / (1024 ** 2), 4),
    "output_path": EXPECTED_LOS_DOCUMENT_PATH.relative_to(PROJECT_ROOT).as_posix(),
}])

final_export_manifest = pd.concat([export_manifest, document_manifest_row], ignore_index=True)
EXPORT_MANIFEST_PATH = OUTPUT_DIR / "export_manifest.csv"
final_export_manifest.to_csv(EXPORT_MANIFEST_PATH, index=False)

for relative_path in final_export_manifest["output_path"]:
    assert (PROJECT_ROOT / relative_path).exists()

display(final_export_manifest)

,file_name,artifact_type,row_count,file_size_mb,output_path
0,environment_versions.csv,Model development output,6,0.0001,outputs/expected_los_model/environment_versions.csv
1,source_record_key_validation.csv,Model development output,1,0.0001,outputs/expected_los_model/source_record_key_validation.csv
2,modeling_population_summary.csv,Model development output,1,0.0002,outputs/expected_los_model/modeling_population_summary.csv
3,feature_specification.csv,Model development output,31,0.0032,outputs/expected_los_model/feature_specification.csv
4,model_candidate_specification.csv,Model development output,12,0.0009,outputs/expected_los_model/model_candidate_specification.csv
5,fold_hospital_summary.csv,Model development output,3,0.0002,outputs/expected_los_model/fold_hospital_summary.csv
6,cross_validation_fold_metrics.csv,Model development output,33,0.0069,outputs/expected_los_model/cross_validation_fold_metrics.csv
7,oof_prediction_validation.csv,Model development output,11,0.0003,outputs/expected_los_model/oof_prediction_validation.csv
8,model_comparison.csv,Model development output,11,0.0028,outputs/expected_los_model/model_comparison.csv
9,model_shortlist.csv,Model development output,3,0.0010,outputs/expected_los_model/model_shortlist.csv


## 24. Clean Temporary Build Artifacts

In [24]:
con.close()

if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()

try:
    WORK_DIR.rmdir()
except OSError:
    pass

print("Temporary expected-LOS modeling artifacts removed.")
print("Aggregate model-development outputs retained successfully.")

Temporary expected-LOS modeling artifacts removed.
Aggregate model-development outputs retained successfully.


# Final Notebook Summary

## Work Completed

- Loaded and validated committed outputs from Notebooks 02–05.
- Scoped modeling to the audited source snapshot.
- Defined an explicit primary feature and leakage contract.
- Excluded LOS-derived, financial, peer-benchmark, hospital, disposition, and procedure fields from the primary model.
- Retained `120 +` LOS observations as observable lower bounds.
- Created a deterministic bounded development sample for local portfolio execution.
- Applied hospital-held-out cross-validation using identical folds across candidates.
- Evaluated statewide, APR-DRG, and APR-DRG × severity mean and median baselines.
- Evaluated Ridge, Poisson, Tweedie, Random Forest, and XGBoost candidates.
- Compared candidates using MAE, median absolute error, RMSE, aggregate calibration, runtime, and improvement over the strong transparent baseline.
- Produced a provisional statistical / ML shortlist rather than selecting a model from one metric alone.
- Produced privacy-safe hospital, APR-DRG, severity, age, payer, and admission-context subgroup diagnostics.
- Omitted groups below the 11-discharge reporting threshold from detailed exports.
- Recorded the Python and modeling-library versions used for execution.
- Evaluated sensitivity to `120 +` LOS records.
- Evaluated hospital-identity and discharge-disposition sensitivity separately from the primary model.
- Defined the Notebook 07 prediction-output contract.
- Exported aggregate modeling diagnostics and generated `docs/expected_los_modeling.md`.

## Key Modeling Decisions

1. The task is retrospective case-mix-adjusted LOS estimation.
2. The target is `los_days_lower_bound`.
3. Hospital-held-out cross-validation is the primary one-year validation design.
4. Hospital identity and patient disposition are sensitivity-only fields.
5. Procedure code is excluded from the primary model because it may reflect care delivered during the stay.
6. Charges, estimated costs, and descriptive peer expectations are forbidden primary features.
7. All candidates use identical validation folds.
8. APR-DRG × severity mean LOS is the strong conditional-mean baseline.
9. APR-DRG × severity median LOS is retained as a robust descriptive reference.
10. Model promotion is not determined by one metric.
11. `peer_expected_los_days` remains distinct from future `model_predicted_los_days`.
12. Future-year stability remains unproven with the current one-year implementation.
13. Notebook 06 produces a provisional shortlist rather than a final model version.

## Remaining Deferred Work

Notebook 07 must review the shortlist and sensitivity results, decide whether ML improvement justifies complexity, freeze the final feature contract, refit the selected model under the approved full-data strategy, assign a model version, create a scoring timestamp, generate `model_predicted_los_days`, produce coverage and missing-reason fields, validate scoring-output reconciliation, and create the Power BI-ready versioned prediction artifact.

## Scope Boundary

This notebook develops and validates candidate expected-LOS models. It does not create a final production model version, causal claims, or a replacement for the transparent descriptive peer benchmark.